# GLADA: Comprehensive Multi-Dataset Evaluation

**Purpose**: Evaluate ALL trained checkpoints from both semesters across all
standard benchmarks, producing thesis-ready results with bootstrap CIs.

**Designed for overnight unattended execution on Colab.**

## Models Evaluated

| Group | Models | Count |
|---|---|---|
| A. GLADA Ablation | 11 configs, WavLM-Base+, seed=42 | 11 |
| B. Backbone Comparison | Full GLADA on 5 backbones | 6 |
| C. Semester 1 Legacy | DANN-IB/CAAN, frozen backbone | 6 |

## Datasets (5)

| Dataset | Type | Reference |
|---|---|---|
| ASVspoof 2019 LA eval | In-domain | Wang et al., 2020 |
| In-the-Wild (ITW) | Cross-domain (primary) | Muller et al., 2024 |
| ASVspoof 2021 LA eval | Codec/channel shift | Yamagishi et al., 2021 |
| ASVspoof 2021 DF eval | Novel deepfake attacks | Yamagishi et al., 2021 |
| WaveFake | Vocoder diversity | Frank & Schonherr, 2021 |

## Estimated Runtime
- ~5 min per model per dataset (T4 GPU)
- ~23 models x 5 datasets x 5 min = ~10 hours
- Results saved incrementally after each model (safe to interrupt)

In [1]:
# ============================================================================
# Cell 1: Environment Setup
#
# Mount Google Drive for checkpoint and dataset access.
# Verify GPU availability and configure AMP precision.
# ============================================================================
import os
import sys
import json
import time
import datetime
import tarfile
from pathlib import Path
from dataclasses import dataclass, field, asdict

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast
from tqdm.auto import tqdm

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# GPU diagnostics
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu_name} ({gpu_mem:.1f} GB)')
    # Enable TF32 for A100 (2x throughput on matmuls)
    if 'a100' in gpu_name.lower() or 'h100' in gpu_name.lower():
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        print('TF32 enabled for A100/H100')

# AMP dtype: bfloat16 for A100 (no overflow), float16 for T4
AMP_DTYPE = torch.bfloat16 if device.type == 'cuda' and (
    'a100' in torch.cuda.get_device_name(0).lower() or
    'h100' in torch.cuda.get_device_name(0).lower()
) else torch.float16
print(f'AMP dtype: {AMP_DTYPE}')

# Output directory for results
RESULTS_DIR = '/content/drive/MyDrive/Jakob/Experiments/comprehensive_eval'
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Results will be saved to: {RESULTS_DIR}')

Mounted at /content/drive
Device: cuda
GPU: Tesla T4 (15.6 GB)
AMP dtype: torch.float16
Results will be saved to: /content/drive/MyDrive/Jakob/Experiments/comprehensive_eval


## GLADA Package Import

The GLADA package (`Coding/glada/`) is synced to Drive and provides model
architecture, config, and dataset classes. Per Rule 13, all core logic lives
in the package; notebooks are thin orchestration layers.

Dataset classes for ASVspoof 2021 and WaveFake are defined in this notebook
since they are eval-only and not part of the training package.

In [2]:
# ============================================================================
# Cell 2: Import GLADA Package from Drive
#
# The glada package contains the model architecture (GLADAModel),
# configuration (GLADAConfig), and core dataset classes (ASVspoof 2019,
# In-the-Wild). We add it to sys.path to import directly.
# ============================================================================
GLADA_PKG_PATH = '/content/drive/Othercomputers/Jakob's MacBook Air/Thesis/298-deliverables/glada-code'
if GLADA_PKG_PATH not in sys.path:
    sys.path.insert(0, GLADA_PKG_PATH)

# Core imports from glada package
from glada.config import GLADAConfig, BACKBONE_REGISTRY, set_seed
from glada.models.glada_model import GLADAModel
from glada.datasets.audio import load_audio
from glada.datasets.asvspoof import ASVspoofEvalDataset
from glada.datasets.itw import InTheWildDataset
from glada.evaluation.metrics import compute_eer, compute_eer_with_ci

# Transformers for backbone loading
from transformers import AutoModel

set_seed(42)
print('GLADA package loaded successfully.')
print(f'Available backbones: {list(BACKBONE_REGISTRY.keys())}')

GLADA package loaded successfully.
Available backbones: ['wavlm-large', 'xlsr-300m', 'data2vec-large', 'wav2vec2-robust', 'wavlm-base-plus', 'xlsr-1b']


## Dataset Extraction

Datasets are stored as compressed archives on Google Drive. Colab ephemeral
storage (`/content/data/`) is wiped between sessions, so we extract on demand.
Extraction is skipped if the target directory already has contents.

In [3]:
# ============================================================================
# Cell 3: Extract All Evaluation Datasets
#
# Five standard benchmarks for audio deepfake detection:
# 1. ASVspoof 2019 LA eval -- in-domain test set (Wang et al., 2020)
# 2. In-the-Wild (ITW) -- cross-domain real-world samples (Muller et al., 2024)
# 3. ASVspoof 2021 LA eval -- codec/channel shift (Yamagishi et al., 2021)
# 4. ASVspoof 2021 DF eval -- novel deepfake attacks (Yamagishi et al., 2021)
# 5. WaveFake -- vocoder diversity benchmark (Frank & Schonherr, 2021)
# ============================================================================

LOCAL_ROOT = '/content/data'
os.makedirs(LOCAL_ROOT, exist_ok=True)

# Archive paths on Google Drive
DRIVE_BASE = '/content/drive/MyDrive/Jakob/Datasets'
DRIVE_ARCHIVES = {
    'asvspoof19':   os.path.join(DRIVE_BASE, 'LA.tar'),
    'itw':          os.path.join(DRIVE_BASE, 'release_in_the_wild.tar'),
    'asvspoof21la': os.path.join(DRIVE_BASE, 'ASVspoof2021_LA_eval.tar.gz'),
    'asvspoof21df': os.path.join(DRIVE_BASE, 'ASVspoof2021_DF_eval.tar.gz'),
    'wavefake':     os.path.join(DRIVE_BASE, 'WaveFake.tar.gz'),
}

# Local extraction targets
LOCAL_DIRS = {
    'asvspoof19':   os.path.join(LOCAL_ROOT, 'asvspoof'),
    'itw':          os.path.join(LOCAL_ROOT, 'itw'),
    'asvspoof21la': os.path.join(LOCAL_ROOT, 'asvspoof21_la'),
    'asvspoof21df': os.path.join(LOCAL_ROOT, 'asvspoof21_df'),
    'wavefake':     os.path.join(LOCAL_ROOT, 'wavefake'),
}


def extract_dataset(tar_path, extract_to, name):
    """Extract a tar/tar.gz archive to local storage, skipping if already present."""
    if not os.path.exists(tar_path):
        print(f'  [{name}] Archive not found: {tar_path} -- SKIPPING')
        return False
    os.makedirs(extract_to, exist_ok=True)
    if os.listdir(extract_to):
        print(f'  [{name}] Already extracted at {extract_to}')
        return True
    print(f'  [{name}] Extracting from {tar_path}...')
    mode = 'r:gz' if tar_path.endswith('.gz') else 'r:*'
    with tarfile.open(tar_path, mode) as tar:
        tar.extractall(path=extract_to)
    print(f'  [{name}] Done -> {extract_to}')
    return True


print('Extracting datasets to Colab local storage...')
print('=' * 60)
available_datasets = {}
for key, tar_path in DRIVE_ARCHIVES.items():
    success = extract_dataset(tar_path, LOCAL_DIRS[key], key)
    available_datasets[key] = success

print(f'\nAvailable: {sum(available_datasets.values())}/{len(available_datasets)}')
for k, v in available_datasets.items():
    print(f'  {k:15s}: {"YES" if v else "NO"}')

Extracting datasets to Colab local storage...
  [asvspoof19] Extracting from /content/drive/MyDrive/Jakob/Datasets/LA.tar...


/tmp/ipykernel_3628/106360276.py:65: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=extract_to)


  [asvspoof19] Done -> /content/data/asvspoof
  [itw] Extracting from /content/drive/MyDrive/Jakob/Datasets/release_in_the_wild.tar...


/tmp/ipykernel_3628/106360276.py:65: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=extract_to)


  [itw] Done -> /content/data/itw
  [asvspoof21la] Extracting from /content/drive/MyDrive/Jakob/Datasets/ASVspoof2021_LA_eval.tar.gz...
  [asvspoof21la] Copying ASVspoof2021_LA_eval.tar.gz to local storage...
  [asvspoof21la] Copied to /content/data/ASVspoof2021_LA_eval.tar.gz
  [asvspoof21la] Done -> /content/data/asvspoof21_la
  [asvspoof21la] Cleaned up temporary local copy.
  [asvspoof21df] Archive not found: /content/drive/MyDrive/Jakob/Datasets/ASVspoof2021_DF_eval.tar.gz -- SKIPPING
  [wavefake] Archive not found: /content/drive/MyDrive/Jakob/Datasets/WaveFake.tar.gz -- SKIPPING

Available: 3/5
  asvspoof19     : YES
  itw            : YES
  asvspoof21la   : YES
  asvspoof21df   : NO
  wavefake       : NO


## Additional Dataset Classes

ASVspoof 2021 LA/DF and WaveFake use different protocol formats than
ASVspoof 2019, so they need dedicated dataset classes. These are eval-only
classes (not part of the training pipeline in `Coding/glada/`).

**ASVspoof 2021** (Yamagishi et al., 2021): Post-challenge ground truth
keys are in `trial_metadata.txt` with a different column layout than 2019.

**WaveFake** (Frank & Schonherr, 2021): Directory-structured dataset with
one subdirectory per vocoder. Audio is at 22050 Hz and needs resampling.

In [4]:
# ============================================================================
# Cell 4: Dataset Classes for ASVspoof 2021 and WaveFake
#
# These classes handle protocol parsing for evaluation benchmarks not
# included in the training pipeline. All return a unified dict:
#   {'input_values': Tensor, 'attention_mask': Tensor, 'task_label': Tensor}
#
# Audio loading uses soundfile.read() via load_audio() from the glada
# package (Rule 7: never use torchaudio.load).
# ============================================================================


class ASVspoof2021Dataset(Dataset):
    """ASVspoof 2021 LA or DF evaluation set (Yamagishi et al., 2021).

    The 2021 challenge introduced telephony codec artifacts (LA track) and
    novel deepfake methods (DF track). Ground truth was released post-challenge.

    Protocol format (trial_metadata.txt):
      speaker_id audio_key ... label ...
    Label is determined by 'bonafide' presence in the line.

    The 2021 LA track tests robustness to telephony channel effects
    (codec compression, bandwidth reduction) applied to BOTH bonafide
    and spoofed audio. This represents a realistic deployment scenario.

    The 2021 DF track tests generalization to deepfake methods NOT seen
    during ASVspoof 2019 training (e.g., newer TTS/VC systems).
    """

    def __init__(self, keys_path, audio_dir, max_len=64000, max_samples=None):
        self.audio_dir = Path(audio_dir)
        self.max_len = max_len
        self.samples = []

        with open(keys_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 2:
                    continue
                filename = parts[1]  # audio key is column 1
                label = 1 if 'bonafide' in line else 0
                self.samples.append({'filename': filename, 'label': label})

        if max_samples:
            self.samples = self.samples[:max_samples]

        n_bona = sum(1 for s in self.samples if s['label'] == 1)
        n_spoof = len(self.samples) - n_bona
        print(f'  ASVspoof 2021: {len(self.samples)} samples '
              f'({n_bona} bonafide, {n_spoof} spoof)')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        # Try .flac first, then .wav
        audio_path = self.audio_dir / f"{s['filename']}.flac"
        if not audio_path.exists():
            audio_path = self.audio_dir / f"{s['filename']}.wav"
        wav = load_audio(str(audio_path), self.max_len)
        mask = torch.ones(self.max_len, dtype=torch.long)
        return {
            'input_values': wav,
            'attention_mask': mask,
            'task_label': torch.tensor(s['label'], dtype=torch.long),
        }


class WaveFakeDataset(Dataset):
    """WaveFake benchmark (Frank & Schonherr, 2021).

    Directory structure:
      wavefake/
        ljspeech_full_band_melgan/   # spoof
        ljspeech_hifiGAN/            # spoof
        ljspeech_melgan/             # spoof
        ...
        LJSpeech-1.1/wavs/          # bonafide (original LJSpeech)

    Audio is 22050 Hz mono WAV. Resampled to 16 kHz by load_audio().
    Bonafide samples are from the original LJSpeech-1.1 corpus;
    spoof samples are generated by various neural vocoders.
    """

    def __init__(self, root_dir, max_len=64000, max_samples=None):
        self.max_len = max_len
        self.samples = []
        root = Path(root_dir)

        # Walk directory tree to find audio files
        for subdir in sorted(root.iterdir()):
            if not subdir.is_dir():
                continue
            # Bonafide: LJSpeech-1.1 directory
            is_bonafide = 'ljspeech-1.1' in subdir.name.lower()
            label = 1 if is_bonafide else 0
            # Find all .wav files (may be in subdirectory like wavs/)
            wav_files = list(subdir.rglob('*.wav'))
            for wav_path in wav_files:
                self.samples.append({'path': str(wav_path), 'label': label})

        if max_samples:
            # Stratified sample to keep class ratio
            import random
            random.seed(42)
            random.shuffle(self.samples)
            self.samples = self.samples[:max_samples]

        n_bona = sum(1 for s in self.samples if s['label'] == 1)
        n_spoof = len(self.samples) - n_bona
        print(f'  WaveFake: {len(self.samples)} samples '
              f'({n_bona} bonafide, {n_spoof} spoof)')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        wav = load_audio(s['path'], self.max_len)
        mask = torch.ones(self.max_len, dtype=torch.long)
        return {
            'input_values': wav,
            'attention_mask': mask,
            'task_label': torch.tensor(s['label'], dtype=torch.long),
        }


print('Additional dataset classes defined.')

Additional dataset classes defined.


## Model Loading Functions

Two model types need loading:

1. **GLADAModel** (Semester 2): Unfrozen SSL backbone with hierarchical
   attention, VIB, per-group domain discriminators, etc. Loaded from the
   `glada` package.

2. **Legacy DomainAdaptationModel** (Semester 1): Frozen SSL backbone with
   mean pooling and a simple classifier head. Used for DANN-IB/CAAN
   experiments from the survey paper. Inlined here since it is NOT part of
   the `glada` package (it predates the package architecture).

In [5]:
# ============================================================================
# Cell 5: Model Loading Functions
#
# GLADA checkpoints store: {'model': state_dict, 'epoch': int,
#   'best_eer': float, 'config': dict}
#
# Legacy DANN-IB checkpoints (Semester 1) use the same format but with
# a simpler model architecture (frozen backbone + linear head).
#
# Both model types produce output dicts with 'spoof_logits' key (B, 2)
# used for EER computation via softmax -> P(bonafide | x).
# ============================================================================


def load_glada_model(checkpoint_path, backbone='wavlm-base-plus'):
    """Load a GLADA model checkpoint.

    Reconstructs GLADAConfig from the saved config dict to ensure
    architecture parameters (hidden_dim, num_groups, etc.) match exactly.

    Args:
        checkpoint_path: Path to .pt file on Drive.
        backbone: Fallback backbone key if config missing from checkpoint.

    Returns:
        (model, config_dict): Model in eval mode, config as dict.
    """
    ckpt = torch.load(checkpoint_path, map_location='cpu', weights_only=False)

    # Reconstruct config
    saved_config = ckpt.get('config', {})
    config = GLADAConfig(backbone=saved_config.get('backbone', backbone))
    for k, v in saved_config.items():
        if hasattr(config, k):
            setattr(config, k, v)
    config.__post_init__()

    # Build and load
    model = GLADAModel(config)
    model.load_state_dict(ckpt['model'])
    model.eval()

    epoch = ckpt.get('epoch', '?')
    best_eer = ckpt.get('best_eer', '?')
    print(f'  Loaded GLADA: backbone={config.backbone}, epoch={epoch}, '
          f'EER={best_eer}')
    return model, asdict(config)


# --- Legacy DomainAdaptationModel (Semester 1) ---
# Reconstructed from reference/domain_adaptation_e2e.py for checkpoint loading.
# Architecture: frozen SSL backbone -> mean pool -> classifier head.
# Optional: domain discriminator with GRL (for DANN/DANN-IB/CAAN methods).


class _LegacyGRL(torch.autograd.Function):
    """Gradient Reversal Layer (Ganin et al., JMLR 2016)."""
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.clone()
    @staticmethod
    def backward(ctx, grad):
        return -ctx.alpha * grad, None


class LegacyDomainAdaptationModel(nn.Module):
    """Survey paper model (Semester 1, CS 297).

    Simple architecture used for the DANN-IB/CAAN comparison study:
      frozen SSL backbone -> mean pool over time -> classifier head
    Optional: information bottleneck, domain discriminator with GRL.

    Differs from GLADAModel in:
      - Backbone always frozen (no fine-tuning)
      - Uses only last hidden state (not all layers)
      - Simple mean pooling (no hierarchical attention)
      - No VIB, no CKA, no speaker adversary, no per-group DA
    """

    def __init__(self, model_name, hidden_size, has_domain_disc=True,
                 has_bottleneck=False, bn_dim=16):
        super().__init__()
        self.wavlm = AutoModel.from_pretrained(model_name)
        self.wavlm.feature_extractor._freeze_parameters()
        for p in self.wavlm.parameters():
            p.requires_grad = False

        # Optional information bottleneck (deterministic, not VIB)
        self.has_bottleneck = has_bottleneck
        if has_bottleneck:
            self.bottleneck = nn.Sequential(
                nn.Linear(hidden_size, bn_dim),
                nn.ReLU(),
            )
            cls_input_dim = bn_dim
        else:
            self.bottleneck = None
            cls_input_dim = hidden_size

        self.classifier = nn.Sequential(
            nn.Linear(cls_input_dim, 256), nn.LayerNorm(256),
            nn.GELU(), nn.Dropout(0.1), nn.Linear(256, 2)
        )
        self.domain_disc = nn.Sequential(
            nn.Linear(cls_input_dim, 256), nn.LayerNorm(256),
            nn.GELU(), nn.Dropout(0.1), nn.Linear(256, 2)
        ) if has_domain_disc else None
        self.grl_alpha = 0.0

    def forward(self, input_values, attention_mask=None):
        with torch.no_grad():
            features = self.wavlm(
                input_values, attention_mask=attention_mask
            ).last_hidden_state.mean(1)

        if self.has_bottleneck:
            features = self.bottleneck(features)

        return {'spoof_logits': self.classifier(features), 'features': features}


# HuggingFace model IDs for legacy backbone loading
LEGACY_BACKBONE_MAP = {
    'wavlm_base_plus':  ('microsoft/wavlm-base-plus', 768),
    'wavlm_large':      ('microsoft/wavlm-large', 1024),
    'xlsr_300m':         ('facebook/wav2vec2-xls-r-300m', 1024),
    'wav2vec2_base':     ('facebook/wav2vec2-base-960h', 768),
    'wav2vec2_large':    ('facebook/wav2vec2-large-960h', 1024),
}


def load_legacy_model(checkpoint_path, backbone='wavlm_base_plus'):
    """Load a Semester 1 DomainAdaptationModel checkpoint.

    These checkpoints use underscore-separated backbone names (e.g.,
    'wavlm_base_plus') matching the Sem 1 naming convention.

    Args:
        checkpoint_path: Path to .pt file on Drive.
        backbone: Backbone key (underscore format).

    Returns:
        (model, info_dict): Model in eval mode, metadata dict.
    """
    model_name, hidden = LEGACY_BACKBONE_MAP[backbone]
    ckpt = torch.load(checkpoint_path, map_location='cpu', weights_only=False)

    # Detect checkpoint format
    if isinstance(ckpt, dict) and 'model' in ckpt:
        state_dict = ckpt['model']
        info = {k: v for k, v in ckpt.items() if k != 'model'}
    else:
        state_dict = ckpt
        info = {}

    # Detect architecture from state_dict keys
    has_domain_disc = any('domain_disc' in k for k in state_dict.keys())
    has_bottleneck = any('bottleneck' in k for k in state_dict.keys())

    # Infer bottleneck dim from weight shape
    bn_dim = 16
    if has_bottleneck:
        for k, v in state_dict.items():
            if 'bottleneck.0.weight' in k:
                bn_dim = v.shape[0]
                break

    model = LegacyDomainAdaptationModel(
        model_name, hidden,
        has_domain_disc=has_domain_disc,
        has_bottleneck=has_bottleneck,
        bn_dim=bn_dim,
    )
    model.load_state_dict(state_dict, strict=False)
    model.eval()

    epoch = info.get('epoch', '?')
    best_eer = info.get('best_eer', '?')
    print(f'  Loaded Legacy: backbone={backbone}, bn={has_bottleneck}'
          f'({bn_dim}), epoch={epoch}, EER={best_eer}')
    return model, info


print('Model loading functions defined.')

Model loading functions defined.


## Standalone Evaluator

A lightweight evaluation function that works with any model returning
`{'spoof_logits': Tensor(B, 2)}`. Computes EER with bootstrap 95% CIs
(required for thesis defense per Rule 3).

**EER** (Equal Error Rate): The threshold $\tau^*$ where the False
Positive Rate equals the False Negative Rate:
$$\text{EER} = \text{FPR}(\tau^*) = \text{FNR}(\tau^*)$$
Standard metric for biometric PAD per ISO/IEC 30107-3.

**Bootstrap CI**: Resample the test set $B=1000$ times with replacement,
compute EER on each resample, report 2.5th and 97.5th percentiles.

In [6]:
# ============================================================================
# Cell 6: Standalone Evaluation Engine
#
# Decoupled from GLADAConfig so it works with both GLADA and legacy models.
# Uses AMP autocast for memory efficiency on large eval sets.
#
# EER computation follows ASVspoof evaluation methodology:
#   scores = P(bonafide | x) via softmax on 2-class logits
#   EER = threshold where FPR(tau) = FNR(tau)
#   Bootstrap CI: resample 1000x, report 95% percentile bounds
#
# References:
#   - EER metric: ISO/IEC 30107-3 (biometric PAD evaluation)
#   - Bootstrap CI: Efron & Tibshirani (1993), used in ASVspoof challenges
# ============================================================================


@torch.no_grad()
def evaluate_model_on_dataset(model, loader, device, dataset_name='dataset',
                              amp_dtype=torch.float16, n_bootstrap=1000):
    """Evaluate a model on a single dataset, returning EER with bootstrap CI.

    Args:
        model: Any model with forward() returning {'spoof_logits': (B, 2)}.
        loader: DataLoader yielding {'input_values', 'attention_mask', 'task_label'}.
        device: torch.device.
        dataset_name: Display name for progress bar.
        amp_dtype: AMP precision (bfloat16 for A100, float16 for T4).
        n_bootstrap: Number of bootstrap resamples for CI.

    Returns:
        dict with keys: eer, ci_low, ci_high, auc, acc, n_samples, name.
    """
    model.eval()
    all_scores, all_labels = [], []

    for batch in tqdm(loader, desc=f'Eval {dataset_name}', leave=False):
        inp = batch['input_values'].to(device, non_blocking=True)
        mask = batch['attention_mask'].to(device, non_blocking=True)

        with autocast('cuda', enabled=(device.type == 'cuda'), dtype=amp_dtype):
            out = model(inp, attention_mask=mask)

        # P(bonafide | x) from 2-class softmax
        probs = F.softmax(out['spoof_logits'], dim=1)[:, 1]
        # BFloat16 -> numpy requires float32 cast (known bug fix)
        all_scores.extend(probs.float().cpu().numpy())
        all_labels.extend(batch['task_label'].numpy())

    labels = np.array(all_labels)
    scores = np.array(all_scores)

    metrics = compute_eer_with_ci(labels, scores, n_bootstrap=n_bootstrap)
    metrics['n_samples'] = len(labels)
    metrics['name'] = dataset_name
    return metrics


def evaluate_model_all_datasets(model, eval_loaders, device, model_name='model',
                                 amp_dtype=torch.float16):
    """Evaluate a model on all available datasets.

    Args:
        model: Model instance.
        eval_loaders: Dict[str, DataLoader] mapping dataset names to loaders.
        device: torch.device.
        model_name: For display.
        amp_dtype: AMP precision.

    Returns:
        Dict[str, dict]: {dataset_name: metrics_dict}
    """
    results = {}
    for ds_name, loader in eval_loaders.items():
        try:
            r = evaluate_model_on_dataset(model, loader, device, ds_name, amp_dtype)
            results[ds_name] = r
            print(f'  {ds_name:<15s}: EER={r["eer"]:.2f}% '
                  f'[{r["ci_low"]:.2f}, {r["ci_high"]:.2f}] '
                  f'AUC={r["auc"]:.4f} (n={r["n_samples"]:,})')
        except Exception as e:
            print(f'  {ds_name:<15s}: ERROR - {e}')
            results[ds_name] = {'eer': -1, 'error': str(e)}
    return results


print('Evaluation engine ready.')

Evaluation engine ready.


## Checkpoint Registry

All models to evaluate, organized in three groups. Paths point to
Google Drive. **Verify these paths exist before running the eval loop.**

Checkpoint naming: `ckpt_ep{NN}.pt` for periodic saves, `best_model.pt`
for best-ITW-EER checkpoint. We use the best epoch from training history.

In [27]:
# ============================================================================
# Cell 7: Checkpoint Registry with Auto-Discovery
#
# Google Drive FUSE mount does not support glob wildcards reliably.
# We use os.listdir to walk subdirectories manually instead.
# ============================================================================

DRIVE_EXP = '/content/drive/MyDrive/Jakob/Experiments'


def find_checkpoint(base_dir, ckpt_name):
    """Search for checkpoint file under base_dir, walking subdirs manually.

    Drive FUSE doesn't support glob, so we use os.listdir.
    Returns latest match (sorted) if multiple subdirs contain the file.
    """
    if not os.path.isdir(base_dir):
        return None

    # Try direct: base_dir/ckpt_name
    direct = os.path.join(base_dir, ckpt_name)
    if os.path.exists(direct):
        return direct

    # Try one level deep: base_dir/{subdir}/ckpt_name
    found = []
    for sub in sorted(os.listdir(base_dir)):
        sub_path = os.path.join(base_dir, sub)
        if os.path.isdir(sub_path):
            candidate = os.path.join(sub_path, ckpt_name)
            if os.path.exists(candidate):
                found.append(candidate)
            # Try two levels deep: base_dir/{subdir}/{subsub}/ckpt_name
            for subsub in sorted(os.listdir(sub_path)):
                subsub_path = os.path.join(sub_path, subsub)
                if os.path.isdir(subsub_path):
                    candidate2 = os.path.join(subsub_path, ckpt_name)
                    if os.path.exists(candidate2):
                        found.append(candidate2)

    return found[-1] if found else None  # Latest (sorted alphabetically)


# --- Group A: GLADA Ablation (WavLM-Base-Plus, seed=42) ---
GROUP_A_SPEC = [
    ('full_glada',        'ablation_full_glada',           'best_model.pt',
'wavlm-base-plus'),
    ('no_glada',          'ablation_no_glada',             'best_model.pt',
'wavlm-base-plus'),
    ('no_vib',            'ablation_no_vib',               'best_model.pt',
'wavlm-base-plus'),
    ('no_speaker',        'ablation_no_speaker',           'best_model.pt',
'wavlm-base-plus'),
    ('no_cka',            'ablation_no_cka',               'best_model.pt',
'wavlm-base-plus'),
    ('no_rawboost',       'ablation_no_rawboost',          'best_model.pt',
'wavlm-base-plus'),
    ('no_hier',           'ablation_no_hier',              'best_model.pt',
'wavlm-base-plus'),
    ('global_da_only',    'ablation_global_da_only',       'best_model.pt',
'wavlm-base-plus'),
    ('hierarchical_only', 'ablation_hierarchical_only',    'best_model.pt',
'wavlm-base-plus'),
    ('no_da',             'ablation_no_da',                'best_model.pt',
'wavlm-base-plus'),
    ('backbone_only',     'ablation_backbone_only',        'best_model.pt',
'wavlm-base-plus'),
    ]

# --- Group B: Backbone Comparison (full GLADA, different backbones) ---
GROUP_B_SPEC = [
      ('bb_wavlm-large_s42',  'ablation_full_wavlm-large',       'best_model.pt',
  'wavlm-large'),
      ('bb_wavlm-large_s123', 'ablation_full_wavlm-large_s123',  'best_model.pt',
  'wavlm-large'),
      ('bb_xlsr-300m_s42',    'ablation_full_xlsr-300m_s42',      'best_model.pt',
  'xlsr-300m'),
      ('bb_data2vec-large',   'ablation_full_data2vec-large',     'best_model.pt',
  'data2vec-large'),
      ('bb_wav2vec2-robust',  'ablation_full_wav2vec2-robust',    'best_model.pt',
  'wav2vec2-robust'),
      ]
# --- Group C: Semester 1 Legacy (frozen backbone) ---
GROUP_C_SPEC = [
      ('sem1_wbp_no_adapt', 'wavlm_base_plus__no_adapt__seed42',  'best_model.pt',
  'wavlm_base_plus'),
      ('sem1_wbp_dann_ib',  'wavlm_base_plus__dann_ib__seed42',   'best_model.pt',
  'wavlm_base_plus'),
      ('sem1_wbp_caan',     'wavlm_base_plus__caan__seed42',      'best_model.pt',
  'wavlm_base_plus'),
      ('sem1_wl_dann_ib',   'wavlm_large__dann_ib__seed42',       'best_model.pt',
  'wavlm_large'),
      ('sem1_wl_caan',      'wavlm_large__caan__seed42',          'best_model.pt',
  'wavlm_large'),
      ('sem1_xlsr_dann_ib', 'xlsr_300m__dann_ib__seed42',         'best_model.pt',
  'xlsr_300m'),
  ]

# Resolve all checkpoint paths
ALL_MODELS = []

print('Resolving checkpoint paths...\n')
for group_name, specs, loader_type in [
    ('Group A: GLADA Ablation', GROUP_A_SPEC, 'glada'),
    ('Group B: Backbone Comparison', GROUP_B_SPEC, 'glada'),
    ('Group C: Semester 1 Legacy', GROUP_C_SPEC, 'legacy'),
]:
    print(f'--- {group_name} ---')
    for display_name, exp_dir, ckpt_name, backbone in specs:
        candidates = [os.path.join(DRIVE_EXP, exp_dir)]
        if loader_type == 'legacy':
            candidates += [
                os.path.join(DRIVE_EXP, 'dann_experiments', exp_dir),
                os.path.join(DRIVE_EXP, 'dann_experiments', 'exp_C_comprehensive', exp_dir),
                os.path.join(DRIVE_EXP, 'ablation_study', exp_dir),
            ]

        resolved = None
        for base in candidates:
            resolved = find_checkpoint(base, ckpt_name)
            if resolved:
                break

        if resolved:
            ALL_MODELS.append((display_name, resolved, loader_type, backbone))
            print(f'  {display_name:<30s} FOUND: {resolved}')
        else:
            ALL_MODELS.append((display_name, 'NOT_FOUND', loader_type, backbone))
            print(f'  {display_name:<30s} NOT FOUND')
    print()

n_found = sum(1 for _, p, _, _ in ALL_MODELS if p != 'NOT_FOUND')
print(f'Checkpoints found: {n_found}/{len(ALL_MODELS)}')

Resolving checkpoint paths...

--- Group A: GLADA Ablation ---
  full_glada                     FOUND: /content/drive/MyDrive/Jakob/Experiments/ablation_full_glada/20260303_063553/best_model.pt
  no_glada                       FOUND: /content/drive/MyDrive/Jakob/Experiments/ablation_no_glada/20260303_090542/best_model.pt
  no_vib                         FOUND: /content/drive/MyDrive/Jakob/Experiments/ablation_no_vib/20260303_111736/best_model.pt
  no_speaker                     FOUND: /content/drive/MyDrive/Jakob/Experiments/ablation_no_speaker/20260303_133649/best_model.pt
  no_cka                         FOUND: /content/drive/MyDrive/Jakob/Experiments/ablation_no_cka/20260303_160528/best_model.pt
  no_rawboost                    FOUND: /content/drive/MyDrive/Jakob/Experiments/ablation_no_rawboost/20260303_181937/best_model.pt
  no_hier                        FOUND: /content/drive/MyDrive/Jakob/Experiments/ablation_no_hier/20260303_205457/best_model.pt
  global_da_only                

In [24]:
import os

# We KNOW this file exists from your ls output
test = '/content/drive/MyDrive/Jakob/Experiments/ablation_full_glada/20260303_063553/best_model.pt'
print(f'exists: {os.path.exists(test)}')
print(f'parent exists: {os.path.isdir(os.path.dirname(test))}')
print(f'grandparent exists: {os.path.isdir(os.path.dirname(os.path.dirname(test)))}')

# Check if listdir works on the experiment dir
exp = '/content/drive/MyDrive/Jakob/Experiments/ablation_full_glada'
print(f'\nlistdir({exp}):')
print(os.listdir(exp))

# Check if it's a string encoding issue
for item in os.listdir(exp):
    print(f'  repr: {repr(item)}, isdir: {os.path.isdir(os.path.join(exp, item))}')

exists: True
parent exists: True
grandparent exists: True

listdir(/content/drive/MyDrive/Jakob/Experiments/ablation_full_glada):
['20260303_063553']
  repr: '20260303_063553', isdir: True


In [26]:
import os
base = '/content/drive/MyDrive/Jakob/Experiments'
# Look for anything with "dann" or "wavlm_base_plus__" in the name
for x in sorted(os.listdir(base)):
    if 'dann' in x.lower() or 'seed42' in x or 'no_adapt' in x or 'caan' in x:
        print(x)

coral_linear_standard_dlr1.0_gp0_ls0.0_seed42
dann_delayed_standard_dlr1.0_gp0_ls0.0_seed42
dann_delayed_strong_dlr3.0_gp0_ls0.0_seed42
dann_delayed_strong_dlr3.0_gp1_ls0.1_seed42
dann_exponential_standard_dlr1.0_gp0_ls0.0_seed42
dann_exponential_strong_dlr3.0_gp0_ls0.0_seed42
dann_exponential_strong_dlr3.0_gp1_ls0.1_seed42
dann_linear_standard_dlr1.0_gp0_ls0.0_seed42
dann_linear_strong_dlr3.0_gp0_ls0.0_seed123
dann_linear_strong_dlr3.0_gp0_ls0.0_seed42
dann_linear_strong_dlr3.0_gp1_ls0.1_seed42
glada_wavlm-large_seed42
hierda_wavlm-large_seed42
mmd_linear_standard_dlr1.0_gp0_ls0.0_seed42
none_linear_standard_dlr1.0_gp0_ls0.0_seed42


In [21]:
import os

base = '/content/drive/MyDrive/Jakob/Experiments'
print('=== Top level ===')
for x in sorted(os.listdir(base))[:30]:
    print(f'  {x}')

# Pick one experiment and drill down
test_dir = os.path.join(base, 'ablation_full_glada')
if os.path.isdir(test_dir):
    print(f'\n=== Inside ablation_full_glada ===')
    for x in sorted(os.listdir(test_dir)):
        full = os.path.join(test_dir, x)
        print(f'  {x}/' if os.path.isdir(full) else f'  {x}')
        if os.path.isdir(full):
            for y in sorted(os.listdir(full))[:10]:
                full2 = os.path.join(full, y)
                print(f'    {y}/' if os.path.isdir(full2) else f'    {y}')
                if os.path.isdir(full2):
                    for z in sorted(os.listdir(full2))[:10]:
                        print(f'      {z}')
else:
    print(f'\n{test_dir} does not exist!')
    # Try to find anything with "ablation" in name
    for x in os.listdir(base):
        if 'ablation' in x.lower():
            print(f'  Found: {x}')

=== Top level ===
  ablation
  ablation_backbone_only
  ablation_full_data2vec-large
  ablation_full_glada
  ablation_full_wav2vec2-robust
  ablation_full_wavlm-large
  ablation_full_wavlm-large_s123
  ablation_full_wavlm-large_s2026
  ablation_full_xlsr-300m
  ablation_full_xlsr-300m_s42
  ablation_global_da_only
  ablation_hierarchical_only
  ablation_no_cka
  ablation_no_da
  ablation_no_glada
  ablation_no_hier
  ablation_no_rawboost
  ablation_no_speaker
  ablation_no_vib
  ablation_study
  comprehensive_eval
  coral_linear_standard_dlr1.0_gp0_ls0.0_seed42
  dann_delayed_standard_dlr1.0_gp0_ls0.0_seed42
  dann_delayed_strong_dlr3.0_gp0_ls0.0_seed42
  dann_delayed_strong_dlr3.0_gp1_ls0.1_seed42
  dann_exponential_standard_dlr1.0_gp0_ls0.0_seed42
  dann_exponential_strong_dlr3.0_gp0_ls0.0_seed42
  dann_exponential_strong_dlr3.0_gp1_ls0.1_seed42
  dann_linear_standard_dlr1.0_gp0_ls0.0_seed42
  dann_linear_strong_dlr3.0_gp0_ls0.0_seed123

=== Inside ablation_full_glada ===
  20260303_

## Create Evaluation Dataloaders

Build DataLoaders for all available evaluation datasets. Datasets not found
on disk are skipped with a warning (graceful degradation).

Eval batch size is larger than training since no gradients are computed.

In [28]:
# ============================================================================
# Cell 8: Create Evaluation DataLoaders
#
# Uses the dataset classes from the glada package (ASVspoof 2019, ITW)
# and the notebook-defined classes (ASVspoof 2021, WaveFake).
#
# All loaders use:
#   - shuffle=False (reproducible ordering)
#   - num_workers=0 (Colab multiprocessing issues)
#   - pin_memory=True (faster CPU->GPU transfer)
#   - max_audio_len=64000 (4 seconds at 16 kHz)
# ============================================================================

MAX_LEN = 64000  # 4 seconds at 16 kHz
EVAL_BATCH_SIZE = 48  # No gradients -> can use larger batches
_eval_cfg = GLADAConfig(backbone='wavlm-base-plus')
_eval_cfg.max_audio_len = MAX_LEN
eval_loaders = {}

# --- ASVspoof 2019 LA eval (in-domain test) ---
ASV19_PROTO = '/content/data/asvspoof/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt'
ASV19_AUDIO = '/content/data/asvspoof/LA/ASVspoof2019_LA_eval/flac'
if os.path.exists(ASV19_PROTO) and os.path.isdir(ASV19_AUDIO):
  print('Loading ASVspoof 2019 LA eval...')
  _eval_cfg = GLADAConfig(backbone='wavlm-base-plus')
  _eval_cfg.max_audio_len = MAX_LEN
  eval_loaders['2019_LA'] = DataLoader(
  ASVspoofEvalDataset(ASV19_PROTO, ASV19_AUDIO, _eval_cfg),
  batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
else:
    print(f'WARNING: ASVspoof 2019 LA not found. Skipping.')

# --- In-the-Wild (primary cross-domain benchmark) ---
ITW_META = '/content/data/itw/release_in_the_wild/meta.csv'
ITW_AUDIO = '/content/data/itw/release_in_the_wild'
if os.path.exists(ITW_META):
  print('Loading In-the-Wild...')
  _itw_cfg = GLADAConfig(backbone='wavlm-base-plus')
  _itw_cfg.max_audio_len = MAX_LEN
  eval_loaders['ITW'] = DataLoader(
      InTheWildDataset(ITW_META, ITW_AUDIO, _itw_cfg, use_labels=True),
      batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
else:
    print(f'WARNING: ITW not found. Skipping.')

# --- ASVspoof 2021 LA eval (codec shift) ---
ASV21_LA_KEYS = '/content/data/asvspoof21_la/keys/LA/CM/trial_metadata.txt'
ASV21_LA_AUDIO = '/content/data/asvspoof21_la/ASVspoof2021_LA_eval/flac'
if os.path.exists(ASV21_LA_KEYS) and os.path.isdir(ASV21_LA_AUDIO):
    print('Loading ASVspoof 2021 LA eval...')
    eval_loaders['2021_LA'] = DataLoader(
        ASVspoof2021Dataset(ASV21_LA_KEYS, ASV21_LA_AUDIO, max_len=MAX_LEN),
        batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
else:
    print(f'WARNING: ASVspoof 2021 LA not found. Skipping.')

# --- ASVspoof 2021 DF eval (novel attacks) ---
ASV21_DF_KEYS = '/content/data/asvspoof21_df/keys/DF/CM/trial_metadata.txt'
ASV21_DF_AUDIO = '/content/data/asvspoof21_df/ASVspoof2021_DF_eval/flac'
if os.path.exists(ASV21_DF_KEYS) and os.path.isdir(ASV21_DF_AUDIO):
    print('Loading ASVspoof 2021 DF eval...')
    eval_loaders['2021_DF'] = DataLoader(
        ASVspoof2021Dataset(ASV21_DF_KEYS, ASV21_DF_AUDIO, max_len=MAX_LEN),
        batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
else:
    print(f'WARNING: ASVspoof 2021 DF not found. Skipping.')

# --- WaveFake (vocoder diversity) ---
WAVEFAKE_ROOT = '/content/data/wavefake'
if os.path.isdir(WAVEFAKE_ROOT) and os.listdir(WAVEFAKE_ROOT):
    print('Loading WaveFake...')
    eval_loaders['WaveFake'] = DataLoader(
        WaveFakeDataset(WAVEFAKE_ROOT, max_len=MAX_LEN),
        batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
else:
    print(f'WARNING: WaveFake not found. Skipping.')

print(f'\nEvaluation datasets loaded: {list(eval_loaders.keys())} ({len(eval_loaders)} total)')
if len(eval_loaders) == 0:
    raise RuntimeError('No evaluation datasets found! Check extraction paths above.')

Loading ASVspoof 2019 LA eval...
  ASVspoof Eval: 71237 samples
Loading In-the-Wild...
  ITW: 31779 samples (labeled returned)

Evaluation datasets loaded: ['2019_LA', 'ITW'] (2 total)


## Run Comprehensive Evaluation

Iterates over all models in the registry, evaluating each on all available
datasets. **Results are saved incrementally** after each model to
`comprehensive_eval/results.json` on Drive, so partial runs are preserved
if the notebook times out.

Each model is loaded, evaluated, then deleted from GPU memory to avoid OOM.

In [29]:
# ============================================================================
# Cell 9: Main Evaluation Loop (designed for overnight unattended execution)
#
# For each model:
#   1. Check if checkpoint exists on Drive
#   2. Check if already evaluated (skip to avoid re-doing work on restart)
#   3. Load model to GPU
#   4. Evaluate on all datasets
#   5. Save results incrementally to Drive
#   6. Free GPU memory
#
# Runtime estimate: ~5 min per model per dataset on T4.
# Total: ~23 models x 5 datasets = ~115 evaluations, ~10 hours.
# ============================================================================

RESULTS_FILE = os.path.join(RESULTS_DIR, 'results.json')

# Load existing results (for resume-after-timeout)
if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE) as f:
        all_results = json.load(f)
    print(f'Loaded {len(all_results)} existing results from previous run.')
else:
    all_results = {}

start_time = time.time()
n_total = len(ALL_MODELS)
n_done = 0
n_skipped = 0
n_failed = 0

for i, (name, ckpt_path, loader_type, backbone) in enumerate(ALL_MODELS):
    print(f'\n{"="*70}')
    print(f'[{i+1}/{n_total}] {name}')
    print(f'{"="*70}')

    # Skip if already evaluated with all current datasets
    if name in all_results:
        existing_datasets = set(all_results[name].get('datasets', {}).keys())
        needed_datasets = set(eval_loaders.keys())
        if needed_datasets.issubset(existing_datasets):
            print(f'  Already evaluated on all datasets. Skipping.')
            n_skipped += 1
            continue
        else:
            missing = needed_datasets - existing_datasets
            print(f'  Partially evaluated. Missing: {missing}. Re-evaluating all.')

    # Check checkpoint exists
    if not os.path.exists(ckpt_path):
        print(f'  Checkpoint not found: {ckpt_path}')
        print(f'  SKIPPING.')
        n_failed += 1
        continue

    # Load model
    try:
        if loader_type == 'glada':
            model, config_info = load_glada_model(ckpt_path, backbone)
        elif loader_type == 'legacy':
            model, config_info = load_legacy_model(ckpt_path, backbone)
        else:
            raise ValueError(f'Unknown loader type: {loader_type}')
        model = model.to(device)
    except Exception as e:
        print(f'  LOAD ERROR: {e}')
        n_failed += 1
        continue

    # Evaluate on all datasets
    try:
        model_results = evaluate_model_all_datasets(
            model, eval_loaders, device, name, amp_dtype=AMP_DTYPE)
    except Exception as e:
        print(f'  EVAL ERROR: {e}')
        import traceback
        traceback.print_exc()
        model_results = {'error': str(e)}
        n_failed += 1

    # Store results
    all_results[name] = {
        'checkpoint': ckpt_path,
        'loader_type': loader_type,
        'backbone': backbone,
        'datasets': model_results,
        'timestamp': datetime.datetime.now().isoformat(),
    }

    # Save incrementally to Drive
    with open(RESULTS_FILE, 'w') as f:
        json.dump(all_results, f, indent=2, default=str)

    # Free GPU memory
    del model
    torch.cuda.empty_cache()

    n_done += 1
    elapsed = time.time() - start_time
    per_model = elapsed / max(n_done, 1)
    remaining = per_model * (n_total - i - 1)
    print(f'\n  Elapsed: {elapsed/60:.1f} min | '
          f'Est. remaining: {remaining/60:.1f} min | '
          f'Done: {n_done}, Skipped: {n_skipped}, Failed: {n_failed}')

# Final summary
total_elapsed = time.time() - start_time
print(f'\n\n{"="*70}')
print(f'EVALUATION COMPLETE')
print(f'{"="*70}')
print(f'Total time: {total_elapsed/60:.1f} min ({total_elapsed/3600:.1f} hours)')
print(f'Models evaluated: {n_done}')
print(f'Models skipped (already done): {n_skipped}')
print(f'Models failed: {n_failed}')
print(f'Results saved to: {RESULTS_FILE}')


[1/22] full_glada


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/378M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

  Loaded GLADA: backbone=wavlm-base-plus, epoch=3, EER=19.636561007871332


... (131 outputs left out)

Eval 2019_LA:   0%|          | 0/1485 [00:00<?, ?it/s]

  2019_LA        : ERROR - Input contains NaN.


Eval ITW:   0%|          | 0/663 [00:00<?, ?it/s]

  ITW            : EER=21.98% [21.46, 22.43] AUC=0.8435 (n=31,779)

  Elapsed: 471.0 min | Est. remaining: 176.6 min | Done: 16, Skipped: 0, Failed: 0

[17/22] sem1_wbp_no_adapt
  Checkpoint not found: NOT_FOUND
  SKIPPING.

[18/22] sem1_wbp_dann_ib
  Checkpoint not found: NOT_FOUND
  SKIPPING.

[19/22] sem1_wbp_caan
  Checkpoint not found: NOT_FOUND
  SKIPPING.

[20/22] sem1_wl_dann_ib
  Checkpoint not found: NOT_FOUND
  SKIPPING.

[21/22] sem1_wl_caan
  Checkpoint not found: NOT_FOUND
  SKIPPING.

[22/22] sem1_xlsr_dann_ib
  Checkpoint not found: NOT_FOUND
  SKIPPING.


EVALUATION COMPLETE
Total time: 471.0 min (7.8 hours)
Models evaluated: 16
Models skipped (already done): 0
Models failed: 6
Results saved to: /content/drive/MyDrive/Jakob/Experiments/comprehensive_eval/results.json


## Results Summary

Load and display all results in table format. Produces both Markdown
(for notebook display) and LaTeX (for thesis).

In [30]:
# ============================================================================
# Cell 10: Results Summary Tables
#
# Formats results from the incremental JSON into publication-ready tables.
# Reports EER with 95% bootstrap CI for each model x dataset combination.
#
# Two output formats:
#   - Markdown table (for notebook display and Craft/Notion sharing)
#   - LaTeX table (for thesis Chapter 5, paste into chap5.tex)
# ============================================================================

# Reload results (in case this cell is run separately)
with open(RESULTS_FILE) as f:
    all_results = json.load(f)

# Dataset order for tables
DS_ORDER = ['2019_LA', 'ITW', '2021_LA', '2021_DF', 'WaveFake']
DS_DISPLAY = {'2019_LA': '19 LA', 'ITW': 'ITW', '2021_LA': '21 LA',
              '2021_DF': '21 DF', 'WaveFake': 'WFake'}

# --- Markdown Table ---
print('### Markdown Table')
print()
header = '| Model | ' + ' | '.join(DS_DISPLAY.get(d, d) for d in DS_ORDER) + ' |'
sep = '|---|' + '---|' * len(DS_ORDER)
print(header)
print(sep)

for name in [m[0] for m in ALL_MODELS]:
    if name not in all_results:
        continue
    row = f'| {name} |'
    datasets = all_results[name].get('datasets', {})
    for ds in DS_ORDER:
        if ds in datasets and isinstance(datasets[ds], dict) and 'eer' in datasets[ds]:
            r = datasets[ds]
            if r['eer'] >= 0:
                margin = (r.get('ci_high', r['eer']) - r.get('ci_low', r['eer'])) / 2
                row += f' {r["eer"]:.2f} +/-{margin:.2f} |'
            else:
                row += f' ERROR |'
        else:
            row += ' - |'
    print(row)

# --- LaTeX Table ---
print()
print()
print('### LaTeX Table (copy to thesis)')
print()
print(r'\begin{table}[ht]')
print(r'\centering')
print(r'\caption{Comprehensive evaluation results (EER \%, lower is better). 95\% bootstrap CI in parentheses.}')
print(r'\label{tab:comprehensive_eval}')
col_spec = 'l' + 'c' * len(DS_ORDER)
print(f'\\begin{{tabular}}{{{col_spec}}}')
print(r'\toprule')
print('Model & ' + ' & '.join(DS_DISPLAY.get(d, d) for d in DS_ORDER) + r' \\')
print(r'\midrule')

for name in [m[0] for m in ALL_MODELS]:
    if name not in all_results:
        continue
    display_name = name.replace('_', '\\_')
    cells = [display_name]
    datasets = all_results[name].get('datasets', {})
    for ds in DS_ORDER:
        if ds in datasets and isinstance(datasets[ds], dict) and 'eer' in datasets[ds]:
            r = datasets[ds]
            if r['eer'] >= 0:
                margin = (r.get('ci_high', r['eer']) - r.get('ci_low', r['eer'])) / 2
                cells.append(f'{r["eer"]:.2f} ($\\pm${margin:.2f})')
            else:
                cells.append('--')
        else:
            cells.append('--')
    print(' & '.join(cells) + r' \\')

print(r'\bottomrule')
print(r'\end{tabular}')
print(r'\end{table}')

print(f'\n\nTotal models with results: {len(all_results)}')
print(f'Results file: {RESULTS_FILE}')

### Markdown Table

| Model | 19 LA | ITW | 21 LA | 21 DF | WFake |
|---|---|---|---|---|---|
| full_glada | 1.04 +/-0.13 | 19.71 +/-0.46 | - | - | - |
| no_glada | 1.36 +/-0.14 | 14.89 +/-0.40 | - | - | - |
| no_vib | 0.72 +/-0.13 | 14.58 +/-0.39 | - | - | - |
| no_speaker | 0.93 +/-0.15 | 13.48 +/-0.39 | - | - | - |
| no_cka | 1.07 +/-0.14 | 12.98 +/-0.37 | - | - | - |
| no_rawboost | 1.05 +/-0.12 | 16.66 +/-0.43 | - | - | - |
| no_hier | 1.15 +/-0.13 | 17.47 +/-0.42 | - | - | - |
| global_da_only | 1.42 +/-0.14 | 14.99 +/-0.41 | - | - | - |
| hierarchical_only | 1.02 +/-0.13 | 13.28 +/-0.35 | - | - | - |
| no_da | 0.80 +/-0.11 | 14.20 +/-0.40 | - | - | - |
| backbone_only | 0.62 +/-0.10 | 13.60 +/-0.39 | - | - | - |
| bb_wavlm-large_s42 | 4.08 +/-0.11 | 12.64 +/-0.36 | - | - | - |
| bb_wavlm-large_s123 | 1.28 +/-0.11 | 15.93 +/-0.41 | - | - | - |
| bb_xlsr-300m_s42 | 0.71 +/-0.08 | 18.40 +/-0.45 | - | - | - |
| bb_data2vec-large | 1.68 +/-0.12 | 18.14 +/-0.45 | - | - | - |
| bb_wav2

## Download Results Locally

After the overnight run, copy the results JSON to your local machine
for visualization and thesis writing.

In [31]:
# ============================================================================
# Cell 11: Copy results to a convenient location and display path
# ============================================================================

print(f'Results saved at: {RESULTS_FILE}')
print(f'\nTo copy to your local machine:')
print(f'  1. Download from Google Drive: Jakob/Experiments/comprehensive_eval/results.json')
print(f'  2. Place at: ~/Desktop/Thesis/Coding/Experiments/comprehensive_eval/results.json')
print(f'\nOr use Colab file download:')

try:
    from google.colab import files
    files.download(RESULTS_FILE)
except:
    print('  (download only works in interactive mode, not during overnight runs)')

Results saved at: /content/drive/MyDrive/Jakob/Experiments/comprehensive_eval/results.json

To copy to your local machine:
  1. Download from Google Drive: Jakob/Experiments/comprehensive_eval/results.json
  2. Place at: ~/Desktop/Thesis/Coding/Experiments/comprehensive_eval/results.json

Or use Colab file download:


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>